In [ ]:
from workflow.scripts.utils import read_list_input_paths
from pyclim_noresm.general_util_funcs import global_avg
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import xarray as xr
import statsmodels.api as sm
# import seaborn

In [ ]:
def get_forcing_value(dfs, variable):
    return {model: df.get('diff').get(variable, np.nan) for model, df in dfs.items()}


with open('workflow/input_data/refractive_indicies_550nm.yaml') as f:
    
    dust_optics = yaml.safe_load(f)


In [ ]:
dfs = {p.split('_')[-1].split('.')[0]: pd.read_csv(p,index_col=0) for p in snakemake.input.erfs}

In [ ]:
atmabs = get_forcing_value(dfs, 'atmabs')
atmabs_sw = get_forcing_value(dfs, 'atmabsSW')
DirectEff = get_forcing_value(dfs, snakemake.wildcards.get('variable'))
ERFt = get_forcing_value(dfs, 'ERFt')

In [ ]:
def get_global_value(dsets,variable):
    return {
            model: global_avg(dset.isel(time=slice(1,None)).mean(dim='time').get(variable,np.nan)).values
              for model, dset in dsets.items()}

In [ ]:
dsets_exp = {p.split('/')[-1].split('_')[-2] : xr.open_dataset(p) for p in snakemake.input.exp_data}
dsets_ctrl = {p.split('/')[-1].split('_')[-2] : xr.open_dataset(p) for p in snakemake.input.ctrl_data}


In [ ]:
aaod_exp = get_global_value(dsets_exp,'abs550aer')
aaod_ctrl = get_global_value(dsets_ctrl,'abs550aer')
df_aaod_exp = pd.DataFrame.from_dict(aaod_exp,orient='index',columns=['abs550aer'])
df_aaod_ctrl = pd.DataFrame.from_dict(aaod_ctrl,orient='index',columns=['abs550aer'])

aod_exp = pd.DataFrame.from_dict(get_global_value(dsets_exp,'od550aer'),orient='index',columns=['od550aer'])
aod_ctrl = pd.DataFrame.from_dict(get_global_value(dsets_ctrl,'od550aer'),orient='index',columns=['od550aer'])

df_diff_aaod = df_aaod_exp-df_aaod_ctrl
df_diff_aod = aod_exp-aod_ctrl

In [ ]:
atmabs = pd.DataFrame.from_dict(atmabs,orient='index',columns=['atmabs'])
atmabs_sw = pd.DataFrame.from_dict(atmabs_sw,orient='index',columns=['atmabs_sw'])
DirectEff = pd.DataFrame.from_dict(DirectEff,orient='index',columns=[snakemake.wildcards.get('variable')])
ERFt = pd.DataFrame.from_dict(ERFt,orient='index',columns=['ERFt'])


In [ ]:
dust_optics = pd.DataFrame.from_dict(dust_optics,orient='index')

In [ ]:
df = pd.concat([df_diff_aod,df_diff_aaod,atmabs,atmabs_sw,DirectEff,ERFt, dust_optics],axis=1)

In [ ]:
model_order=['EC-Earth3-AerChem','MPI-ESM-1-2-HAM','NorESM2-LM','IPSL-CM6A-LR-INCA','UKESM1-0-LL','CNRM-ESM2-1','GFDL-ESM4','MIROC6','GISS-E2-1-G']

In [ ]:
df = df.reindex(model_order)

In [ ]:

def plot_fig_robust2(df, forcingv):
    # Drop specified rows from the dataframe
    df = df.drop(['MIROC6', 'GISS-E2-1-G'], axis=0)
    y = forcingv
    x = 'od550aer'
    
    # Set up the plot
    fig, ax = plt.subplots(figsize=(4*1.5, 3.6*1.51)) 
    cmap = mpl.cm.get_cmap('Blues', 13)
    norm = mpl.colors.Normalize(vmin=0.0001, vmax=0.0012)
    
    # Create a scatter plot
    df.plot.scatter(y=y, x=x, ax=ax, s=50, c='abs550aer', colorbar=False, norm=norm,
                    colormap='Blues', zorder=100)

    # Annotate points on the scatter plot
    for k, v in df.iterrows():
        xy = (v[x], v[y])
        ax.annotate(f'{k}', xy, xytext=(5, -5), textcoords='offset points', fontsize=8, zorder=200)

    # Calculate slopes with respect to the origin (0,0)
    slopes = df[y] / df[x]
    slope = np.median(slopes)

    # Calculating coordinates for the regression line through zero
    xl = [0.045, 0.00]
    yl = [slope * 0.045, 0.00]  # Because line goes through the origin
    
    # Plot the regression line
    ax.plot(xl, yl, '--', color='red', linewidth=3, label=f"y={slope:.3f}x")
    ax.legend(loc=4, frameon=False)
    
    # Tabulate some data on the plot
    dftab = df[['complex']].round(decimals=4)    
    dftab = dftab.rename(columns={'complex': '$n_i$'})
    pd.plotting.table(ax=ax, data=dftab[['$n_i$']], loc=3, bbox=[0.35, 0.58, 0.12, 0.4])
    
    # Set the colorbar
    cax = fig.add_axes([0.94,0.2,0.02,0.62])
    fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=cmap), cax=cax, extend='max', label='DOD 550m')
    
    # Set plot limits and labels
    ax.set_ylim(-0.65, 0.15)
    ax.set_xlim(0, 0.04)
    ax.set_xlabel('DOD 550nm')
    ax.axes.invert_xaxis()
    ax.set_ylabel("Direct DuERF ($\mathrm{Wm}^{-2}$)")
    corr_df = df.corr()
    ax.text(0.64,0.32, "$r_{od550,Direct DuERF}$ :" + f" {corr_df[y]['od550aer']:.2f}\n"+"$r_{abs550,Direct DuERF}$ :" + f" {corr_df[y]['abs550aer']:.2f}", 
            transform=ax.transAxes,     bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))
    
# Assuming df is your DataFrame
# plot_fig(df)


In [ ]:
def calculate_explained_variability(df):
    # Prepare the data
    y_col = snakemake.wildcards.get('variable')
    x1 = 'od550aer'
    x2 = 'abs550aer'
    y_vals = df[y_col]
    
    # Model 1: OLS with only od550aer
    X1 = sm.add_constant(df[[x1]])
    model1 = sm.OLS(y_vals, X1).fit()
    r_squared_x1 = model1.rsquared
    
    # Model 2: OLS with only abs550aer
    X2 = sm.add_constant(df[[x2]])
    model2 = sm.OLS(y_vals, X2).fit()
    r_squared_x2 = model2.rsquared
    
    # Model 3: OLS with both od550aer and abs550aer
    X_full = sm.add_constant(df[[x1, x2]])
    model_full = sm.OLS(y_vals, X_full).fit()
    r_squared_full = model_full.rsquared
    
    # Compute the added R-squared for each variable
    od550aer_contribution = r_squared_full - r_squared_x2
    abs550aer_contribution = r_squared_full - r_squared_x1

    results = {
        'od550aer_contribution': od550aer_contribution,
        'abs550aer_contribution': abs550aer_contribution,
        'full_model_r_squared': r_squared_full
    }

    return results

def plot_fig_multireg(df, forcingv):
    # Perform explained variability calculation

    # Drop specified rows from the dataframe
    df = df.drop(['MIROC6', 'GISS-E2-1-G'], axis=0,errors='ignore')
    y_col = forcingv
    x1 = 'od550aer'
    x2 = 'abs550aer'
    explained_variability = calculate_explained_variability(df)
    
    # Set up the plot
    fig, ax = plt.subplots(figsize=(4*1.5, 3.6*1.51)) 
    cmap = mpl.cm.get_cmap('Blues', 13)
    norm = mpl.colors.Normalize(vmin=0.0001, vmax=0.0012)
    
    # Create a scatter plot
    df.plot.scatter(y=y_col, x=x1, ax=ax, s=50, c=x2, colorbar=False, norm=norm,
                    colormap='Blues', zorder=100)

    # Annotate points on the scatter plot
    for k, v in df.iterrows():
        xy = (v[x1], v[y_col])
        ax.annotate(f'{k}', xy, xytext=(5, -5), textcoords='offset points', fontsize=8, zorder=200)

    # Perform multiple linear regression with statsmodels
    print(df)
    X_full = sm.add_constant(df[[x1, x2]])
    y_vals = df[y_col]
    model = sm.OLS(y_vals, X_full).fit()

    # Extract regression coefficients
    intercept = model.params['const']
    coef_od550 = model.params[x1]
    coef_abs550 = model.params[x2]

    # Define a line based on the regression model
    x_vals = np.linspace(0, 0.04, 100)
    median_abs550 = np.median(df[x2])
    y_pred = intercept + coef_od550 * x_vals + coef_abs550 * median_abs550
    
    # Plot the regression line
    ax.plot(x_vals, y_pred, '--', color='red', linewidth=3, 
            label=f"y={intercept:.3f} + {coef_od550:.3f}*DOD + {coef_abs550:.3f}*DAOD")
    ax.legend(loc=4, frameon=False)
    
    # Tabulate some data on the plot
    dftab = df[['complex']].round(decimals=4)    
    dftab = dftab.rename(columns={'complex': '$n_i$'})
    pd.plotting.table(ax=ax, data=dftab[['$n_i$']], loc=3, bbox=[0.35, 0.58, 0.12, 0.4])
    
    # Set the colorbar
    cax = fig.add_axes([0.94,0.2,0.02,0.62])
    fig.colorbar(mpl.cm.ScalarMappable(norm, cmap=cmap), cax=cax, extend='max', label='$\Delta$ DAOD 550m')
    
    # Set plot limits and labels
    ax.set_ylim(-0.65, 0.15)
    ax.set_xlim(0, 0.04)
    ax.set_xlabel('$\Delta$ DAOD 550nm')
    ax.axes.invert_xaxis()
    if forcingv == 'SWDirectEff':
        ax.set_ylabel("Direct SW DuERF ($\mathrm{Wm}^{-2}$)")
    else:
        ax.set_ylabel("Direct DuERF ($\mathrm{Wm}^{-2}$)")
    # Display R-squared values
    ax.text(0.61, 0.22, f"Total $R^2$: {explained_variability['full_model_r_squared']:.2f}\n"
            f"$R^2$ DOD 550nm: {explained_variability['od550aer_contribution']:.2f}\n"
            f"$R^2$ DAOD 550nm: {explained_variability['abs550aer_contribution']:.2f}", 
            transform=ax.transAxes, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))
# Ensure the necessary imports and data are set up before running this function.


In [ ]:
plot_fig_multireg(df,snakemake.wildcards.get('variable'))

plt.savefig(snakemake.output.absortion_plot, bbox_inches='tight', dpi=300)